# 03. Reliability·Protocol 심화 실습

목표: CRC, segmentation, window, CoAP Blockwise, FEC/ARQ와 DTN scheduling의 trade-off를 수치로 확인한다.

In [1]:
import math
import zlib
from dataclasses import dataclass

def crc32(data: bytes) -> int:
    # CRC는 우연한 오류 탐지용이며 공격자의 위조를 막는 MAC/서명이 아니다.
    return zlib.crc32(data) & 0xffffffff

payload = b'satellite-telemetry-v1'
expected = crc32(payload)
corrupted = bytearray(payload)
corrupted[3] ^= 0b00000100
print(hex(expected), hex(crc32(corrupted)), expected != crc32(corrupted))
assert expected != crc32(corrupted)

0x8f6cbdb2 0x6604b050 True


In [2]:
def segmentation_metrics(payload_bytes, segment_payload, header_bytes, segment_success):
    count = math.ceil(payload_bytes / segment_payload)
    transmitted = payload_bytes + count * header_bytes
    all_success = segment_success ** count
    return count, transmitted, all_success

for size in (64, 128, 256, 512, 1024):
    count, sent, probability = segmentation_metrics(4096, size, 20, 0.99)
    print(f'{size:4d} B: {count:2d}개, overhead={sent-4096:4d} B, 무재전송 전체성공={probability:.4f}')

  64 B: 64개, overhead=1280 B, 무재전송 전체성공=0.5256
 128 B: 32개, overhead= 640 B, 무재전송 전체성공=0.7250
 256 B: 16개, overhead= 320 B, 무재전송 전체성공=0.8515
 512 B:  8개, overhead= 160 B, 무재전송 전체성공=0.9227
1024 B:  4개, overhead=  80 B, 무재전송 전체성공=0.9606


In [3]:
def minimum_window_segments(rate_bps, rtt_s, payload_bytes):
    return math.ceil(rate_bps * rtt_s / (payload_bytes * 8))

cases = [('LEO', 5e6, .030), ('GEO', 5e6, .600), ('Moon', 5e6, 2.6)]
for name, rate, rtt in cases:
    window = minimum_window_segments(rate, rtt, 1024)
    memory = window * 1024
    print(f'{name}: window={window:,} segments, sender buffer≈{memory/1e6:.3f} MB')

LEO: window=19 segments, sender buffer≈0.019 MB
GEO: window=367 segments, sender buffer≈0.376 MB
Moon: window=1,587 segments, sender buffer≈1.625 MB


In [4]:
def coap_blocks(data: bytes, block_size=256):
    allowed = (16, 32, 64, 128, 256, 512, 1024)
    if block_size not in allowed:
        raise ValueError(f'block_size는 {allowed} 중 하나여야 합니다.')
    szx = int(math.log2(block_size)) - 4
    result = []
    for number, offset in enumerate(range(0, len(data), block_size)):
        chunk = data[offset:offset + block_size]
        more = offset + block_size < len(data)
        result.append({'NUM': number, 'M': int(more), 'SZX': szx, 'payload': chunk})
    return result

blocks = coap_blocks(bytes(range(256)) * 5, 256)
print([(b['NUM'], b['M'], b['SZX'], len(b['payload'])) for b in blocks])
assert b''.join(b['payload'] for b in blocks) == bytes(range(256)) * 5

[(0, 1, 4, 256), (1, 1, 4, 256), (2, 1, 4, 256), (3, 1, 4, 256), (4, 0, 4, 256)]


In [5]:
def expected_arq_bytes(payload_bytes, packet_error_rate, header_bytes=20):
    if not 0 <= packet_error_rate < 1:
        raise ValueError('PER은 0 이상 1 미만이어야 합니다.')
    return (payload_bytes + header_bytes) / (1 - packet_error_rate)

def fec_bytes(payload_bytes, code_rate, header_bytes=20):
    if not 0 < code_rate <= 1:
        raise ValueError('code rate는 0 초과 1 이하여야 합니다.')
    return payload_bytes / code_rate + header_bytes

for per in (.01, .1, .3, .5):
    arq = expected_arq_bytes(512, per)
    fec = fec_bytes(512, .75)
    print(f'PER={per:.2f}: ARQ 기대 {arq:.1f} B, rate-3/4 FEC {fec:.1f} B')

# 단순 비교일 뿐 FEC 적용 뒤 PER, ACK, RTT, energy를 생략했다.

PER=0.01: ARQ 기대 537.4 B, rate-3/4 FEC 702.7 B
PER=0.10: ARQ 기대 591.1 B, rate-3/4 FEC 702.7 B
PER=0.30: ARQ 기대 760.0 B, rate-3/4 FEC 702.7 B
PER=0.50: ARQ 기대 1064.0 B, rate-3/4 FEC 702.7 B


In [6]:
@dataclass
class Bundle:
    bundle_id: str
    size: int
    priority: int       # 큰 수가 더 중요
    deadline_s: int
    created_s: int

def schedule_bundles(bundles, now_s, capacity_bytes):
    # 긴급도와 age를 반영하되 실제 mission은 fairness와 destination route도 포함해야 한다.
    def score(bundle):
        slack = max(1, bundle.deadline_s - now_s)
        age = max(0, now_s - bundle.created_s)
        return 1000 * bundle.priority + 10_000 / slack + age / 10
    selected, remaining = [], capacity_bytes
    for bundle in sorted(bundles, key=score, reverse=True):
        if now_s <= bundle.deadline_s and bundle.size <= remaining:
            selected.append(bundle)
            remaining -= bundle.size
    return selected, remaining

queue = [
    Bundle('critical-tm', 1000, 3, 120, 0),
    Bundle('image-1', 9000, 1, 1000, 0),
    Bundle('event', 2000, 2, 80, 40),
    Bundle('old-bulk', 3000, 1, 900, -500),
]
chosen, left = schedule_bundles(queue, now_s=60, capacity_bytes=6000)
print([b.bundle_id for b in chosen], '남은 용량:', left)

['critical-tm', 'event', 'old-bulk'] 남은 용량: 0


## 확장 과제

1. CRC-16-CCITT를 polynomial long division으로 직접 구현하고 알려진 test vector로 검증한다.
2. FEC 뒤 packet error rate를 SNR 함수로 가정하고 latency·energy까지 비교한다.
3. CoAP block ACK가 RTT마다 필요한 모델과 Q-Block 전송 모델을 비교한다.
4. DTN scheduler에 destination별 contact capacity, expiry, custody, replication budget을 추가한다.
5. duplicate command ID cache를 만들고 reboot 전후 persistent journal의 필요성을 실험한다.